# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [3]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked,
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [4]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [32]:
EVENT_NAME = '202410_Landslide_PalosVerdes'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'opera'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 85 .tif files in the S3 bucket.


['drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Apr2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Apr2024_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_April2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Aug2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Aug2024_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_August2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Dec2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Dec2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_December2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Feb20

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


In [8]:
coherence = [i for i in keys if 'coherence' in i]
displacement = [i for i in keys if 'displacement' in i]


In [9]:
config_coherence_up = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW1}/coherence",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

config_coherence_descending = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW1}/coherence/descending",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

config_displacement = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW1}/displacement",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

config_uavsar = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW2}",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

## Configure bucket and paths (no need to create session manually)

In [10]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [14]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [16]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 107
  - Total size: 45.79 GB

📁 Cached files (first 10):
  - drcs_activations/202410_Hurricane_Milton/blackmarble/Con_VJ146A31.tif (45.1 MB)
  - drcs_activations/202410_Hurricane_Milton/blackmarble/Con_VJ146A32.tif (45.1 MB)
  - drcs_activations/202410_Hurricane_Milton/blackmarble/Con_VJ146A33.tif (45.1 MB)
  - drcs_activations/202410_Hurricane_Milton/blackmarble/Con_VJ146A34.tif (45.1 MB)
  - drcs_activations/202410_Hurricane_Milton/blackmarble/VJ146A3.A2024.00.September2024.Mosaic.C2.tif (44.0 MB)
  - drcs_activations/202410_Hurricane_Milton/blackmarble/VNP46A2.A2024284.DNB_BRDF-Corrected_NTL.Mosaic_VNP_C2.tif (44.0 MB)
  - drcs_activations/202410_Hurricane_Milton/blackmarble/VNP46A2.A2024284.QF_Cloud_Mask.Mosaic_VNP_C2.tif (11.0 MB)
  - drcs_activations/202410_Hurricane_Milton/blackmarble/VNP46A2.A2024285.DNB_BRDF-Corrected_NTL.Mosaic_VNP_C2.tif (44.0 MB)
  - drcs_activations/202410_Hurricane_Milton/blackmarble/VNP46A2.A

(107, 49163601015)

In [17]:
coherence_up = [i for i in coherence if 'coherence/PV' in i]
coherence_up

['drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Apr2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Apr2024_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_April2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Aug2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Aug2024_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_August2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Dec2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Dec2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_December2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Feb20

In [18]:
coherence_desc = [i for i in coherence if 'descending' in i]
coherence_desc

['drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_Apr2023_S1D71.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_Apr2024_S1D71.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_April2017-2022_S1D71.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_Aug2023_S1D71.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_Aug2024_S1D71.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_August2017-2022_S1D71.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_Dec2022_S1D71.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_Dec2023_S1D71.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_December2017-20

# Process files

In [24]:
def create_cog_filename_uavsar(f, EVENT_NAME):
    """Create COG filename for UAVSAR files."""
    from pathlib import Path
    
    f2 = Path(f).stem
    fsplit = f2.split('_')
    
    # Original: PV_DISP_20240918_20240930_UAVSAR_T09517
    # Target: EVENT_NAME_PV_DISP_UAVSAR_T09517_d2024-09-18_2024-09-30_day.tif
    
    # Format the dates from YYYYMMDD to YYYY-MM-DD
    date1 = fsplit[2]  # 20240918
    date2 = fsplit[3]  # 20240930
    
    formatted_date1 = f"{date1[:4]}-{date1[4:6]}-{date1[6:8]}"
    formatted_date2 = f"{date2[:4]}-{date2[4:6]}-{date2[6:8]}"
    
    # Reorder components with formatted dates
    cog_filename = f'{EVENT_NAME}_{fsplit[0]}_{fsplit[1]}_{fsplit[4]}_{fsplit[5]}_d{formatted_date1}_{formatted_date2}_day.tif'
    
    return cog_filename
    
filter_str = 'opera/uavsar'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_uavsar(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


Testing WM filename:
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T09517_d2024-09-18_2024-09-30_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T09517_d2024-09-30_2024-10-08_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T09517_d2024-10-08_2024-10-17_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T18527_d2024-09-18_2024-09-30_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T18527_d2024-09-30_2024-10-08_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T18527_d2024-10-08_2024-10-17_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T27558_d2024-09-18_2024-09-30_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T27558_d2024-09-30_2024-10-09_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T27558_d2024-10-09_2024-10-17_day.tif


In [25]:
# # Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_uavsar, 
                                target_dir = "UAVSAR", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T09517_d2024-09-18_2024-09-30_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T09517_d2024-09-30_2024-10-08_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T09517_d2024-10-08_2024-10-17_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T18527_d2024-09-18_2024-09-30_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T18527_d2024-09-30_2024-10-08_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T18527_d2024-10-08_2024-10-17_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T27558_d2024-09-18_2024-09-30_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T27558_d2024-09-30_2024-10-09_day.tif
  202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T27558_d2024-10-09_2024-10-17_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202410_Landslide_PalosVerdes/opera
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/UAVSAR

🌊 Processing Files (Chun

Reading input: /tmp/tmpfgmjhz3k_temp.tif



   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpxxda7klr.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T09517_d2024-09-18_2024-09-30_day.tif
   [MEMORY] Final: 356.1 MB (Change: +60.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T09517_d2024-09-18_2024-09-30_day.tif

[2/9] Processing: drcs_activations/202410_Landslide_PalosVerdes/opera/uavsar/T09517/PV_DISP_20240930_20241008_UAVSAR_T09517.tif
   Output filename: 202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T09517_d2024-09-30_2024-10-08_day.tif
   [MEMORY] Initial: 356.1 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...


...

   [VERIFY] Band 1: min=nan, max=nan, center sample non-zero=398160/398161
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T27558_d2024-09-18_2024-09-30_day.tif
   [MEMORY] Final: 363.3 MB (Change: +3.6 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_PV_DISP_UAV

Reading input: /tmp/tmpcxr7ua_n_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp4j0bvldy.tif


   [VERIFY] Band 1: min=nan, max=nan, center sample non-zero=398160/398161
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T27558_d2024-09-30_2024-10-09_day.tif
   [MEMORY] Final: 363.9 MB (Change: +0.6 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_PV_DISP_UAV

Reading input: /tmp/tmpn93mynzv_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpgr0qdboa.tif


   [VERIFY] Band 1: min=nan, max=nan, center sample non-zero=398160/398161
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/202410_Landslide_PalosVerdes_PV_DISP_UAVSAR_T27558_d2024-10-09_2024-10-17_day.tif
   [MEMORY] Final: 363.9 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_PV_DISP_UAV

In [26]:
keys

['drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Apr2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Apr2024_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_April2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Aug2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Aug2024_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_August2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Dec2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Dec2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_December2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Feb20

In [29]:
def create_cog_filename_cumulative(f, EVENT_NAME):
    """Create COG filename for cumulative displacement files."""
    from pathlib import Path
    import re
    
    f2 = Path(f).stem
    
    # Find the two dates in YYYYMMDD format
    date_pattern = r'(\d{8})_(\d{8})'
    match = re.search(date_pattern, f2)
    
    if match:
        date1 = match.group(1)  # 20160719
        date2 = match.group(2)  # 20230625
        
        # Format dates to YYYY-MM-DD
        formatted_date1 = f"{date1[:4]}-{date1[4:6]}-{date1[6:8]}"
        formatted_date2 = f"{date2[:4]}-{date2[4:6]}-{date2[6:8]}"
        
        # Get everything before the dates
        prefix = f2[:match.start()].rstrip('_')
        # Get everything after the dates
        suffix = f2[match.end():].lstrip('_')
        
        # Reconstruct with formatted dates at the end
        if suffix:
            cog_filename = f'{EVENT_NAME}_{prefix}_{suffix}_d{formatted_date1}_{formatted_date2}_day.tif'
        else:
            cog_filename = f'{EVENT_NAME}_{prefix}_d{formatted_date1}_{formatted_date2}_day.tif'
    else:
        # Fallback if pattern not found
        cog_filename = f'{EVENT_NAME}_{f2}_day.tif'
    
    return cog_filename


filter_str = 'displacement'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_cumulative(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202410_Landslide_PalosVerdes_PV_Cumulative_DISP_TS_S1D71_d2016-07-19_2023-06-25_day.tif
  202410_Landslide_PalosVerdes_PV_Cumulative_DISP_TS_S1A64_d2016-07-07_2023-06-25_day.tif


In [31]:
# # Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_cumulative, 
                                target_dir = "Sentinel-1/displacement", 
                                EVENT_NAME = EVENT_NAME)

Reading input: /tmp/tmpx6vi6gx__temp.tif

Updating dataset tags...
Writing output to: /tmp/tmp51pk9vo7.tif


Testing filenames:
  202410_Landslide_PalosVerdes_PV_Cumulative_DISP_TS_S1D71_d2016-07-19_2023-06-25_day.tif
  202410_Landslide_PalosVerdes_PV_Cumulative_DISP_TS_S1A64_d2016-07-07_2023-06-25_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202410_Landslide_PalosVerdes/opera
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/displacement

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202410_Landslide_PalosVerdes

[1/2] Processing: drcs_activations/202410_Landslide_PalosVerdes/opera/displacement/Descending/PV_Cumulative_DISP_TS_20160719_20230625_S1D71.tif
   Output filename: 202410_Landslide_PalosVerdes_PV_Cumulative_DISP_TS_S1D71_d2016-07-19_2023-06-25_day.tif
   [MEMORY] Initial: 364.1 MB
   [CACHE HIT] Using cached file: data_download/drcs_activations/202410_Landslide_PalosVerdes/opera/displacement/Descending/PV_Cumulative_DISP_TS_20160719_20230625_S1D71.tif
   [REPROJECT] Already in EPS

Reading input: /tmp/tmplrit1qah_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmp3vrysa8w.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_PV_Cumulative_DISP_TS_S1D71_d2016-07-19_2023-06-25_day.tif

[2/2] Processing: drcs_activations/202410_Landslide_PalosVerdes/opera/displacement/PV_Cumulative_DISP_TS_20160707_20230625_S1A64.tif
   Output filename: 202410_Landslide_PalosVerdes_PV_Cumulative_DISP_TS_S1A64_d2016-07-07_2023-06-25_day.tif
   [MEMORY] Initial: 364.1 MB
   [CACHE HIT] Using cached file: data_download/drcs_activations/202410_Landslide_PalosVerdes/opera/displacement/PV_Cumulative_DISP_TS_20160707_20230625_S1A64.tif
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-0.1394132673740387, max=0.3594248592853546, center sample non-zero=2033/3600
            Estimated data coverage: 55.5% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using n

In [34]:
def create_cog_filename_coherence_up(f, EVENT_NAME):
    """Create COG filename for coherence files with proper date formatting."""
    from pathlib import Path
    import re
    
    f2 = Path(f).stem
    fsplit = f2.split('_')
    
    # Extract the date/period part (e.g., 'Apr2023' or 'April2015-2022')
    date_part = fsplit[2]
    
    # Check if it's a range (contains hyphen) or single date
    if '-' in date_part:
        # Handle range format: 'April2015-2022' -> 'April_2015_year_2022'
        # Extract month name and years
        match = re.match(r'([A-Za-z]+)(\d{4})-(\d{4})', date_part)
        if match:
            month = match.group(1)
            year1 = match.group(2)
            year2 = match.group(3)
            date_suffix = f'{month}_{year1}_year_{year2}'
        else:
            date_suffix = date_part  # fallback
    else:
        # Handle single date format: 'Apr2023' -> '2023-04_monthly'
        # Dictionary to convert month abbreviations to numbers
        month_map = {
            'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
            'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08',
            'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
        }
        
        # Extract month and year
        match = re.match(r'([A-Za-z]+)(\d{4})', date_part)
        if match:
            month_abbr = match.group(1)
            year = match.group(2)
            
            # Get month number
            month_num = month_map.get(month_abbr, '00')
            date_suffix = f'{year}-{month_num}_monthly'
        else:
            date_suffix = date_part  # fallback
    
    # Build filename: EVENT_NAME_PV_Avg12dayCoh_S1A64_[date_suffix].tif
    cog_filename = f'{EVENT_NAME}_{fsplit[0]}_{fsplit[1]}_{fsplit[3]}_{date_suffix}.tif'
    
    return cog_filename


filter_str = 'coherence/PV'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_coherence_up(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-04_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2024-04_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_April_2015_year_2022.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-08_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2024-08_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_August_2015_year_2022.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2022-12_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-12_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_December_2015_year_2022.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-02_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2024-02_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_February_2015_year_2022.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-01_monthly.tif
  202410_Landsli

In [35]:
# # Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_coherence_up, 
                                target_dir = "Sentinel-1/coherence", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-04_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2024-04_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_April_2015_year_2022.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-08_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2024-08_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_August_2015_year_2022.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2022-12_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-12_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_December_2015_year_2022.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-02_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2024-02_monthly.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_February_2015_year_2022.tif
  202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-01_monthly.tif
  202410_Landslide

Reading input: /tmp/tmpk6oi35dx_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpcqe54aaa.tif


   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/coherence/202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-04_monthly.tif
   [MEMORY] Final: 364.2 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-04_monthly.tif

[2/37] Processing: drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Apr2024_S1A64.tif
   Output filename: 20

Reading input: /tmp/tmpulmbu6kn_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpsvrdtk75.tif


   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/coherence/202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2024-04_monthly.tif
   [MEMORY] Final: 364.2 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2024-04_monthly.tif

[3/37] Processing: drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_April2015-2022_S1A64.tif
   Output filen

...

   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=0.9898135662078857, center sample non-zero=2035/3600
            Estimated data coverage: 55.6% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/coherence/202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-09_monthly.tif
   [MEMORY] Final: 36

Reading input: /tmp/tmpsmqefr93_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpas2gls2q.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2023-09_monthly.tif

[36/37] Processing: drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Sep2024_S1A64.tif
   Output filename: 202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_2024-09_monthly.tif
   [MEMORY] Initial: 364.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=0.9960275888442993, center sample non-zero=2035/3600
            Estimated data coverage: 55.6% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] 

Reading input: /tmp/tmpqgbcdn3w_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpo7chkmxy.tif


   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=0.9915555715560913, center sample non-zero=2035/3600
            Estimated data coverage: 55.6% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/coherence/202410_Landslide_PalosVerdes_PV_Avg12dayCoh_S1A64_September_2015_year_2022.tif
   [MEMORY] Final: 364.4 MB (Change: +0.0 M

In [37]:
keys

['drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Apr2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Apr2024_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_April2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Aug2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Aug2024_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_August2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Dec2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Dec2023_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_December2015-2022_S1A64.tif',
 'drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/PV_Avg12dayCoh_Feb20

In [38]:

def create_cog_filename_desc(f, EVENT_NAME):
    """Create COG filename for coherence files with proper date formatting."""
    from pathlib import Path
    import re
    
    f2 = Path(f).stem
    fsplit = f2.split('_')
    
    # Check if this is a descending file by looking at the path
    full_path = str(f)
    if 'descending' in full_path:
        direction = 'descending'
    else:
        direction = None  # or 'ascending' if you have those too
    
    # Extract the date/period part (e.g., 'Apr2023' or 'April2015-2022')
    date_part = fsplit[2]
    
    # Check if it's a range (contains hyphen) or single date
    if '-' in date_part:
        # Handle range format: 'April2015-2022' -> 'April_2015_year_2022'
        # Extract month name and years
        match = re.match(r'([A-Za-z]+)(\d{4})-(\d{4})', date_part)
        if match:
            month = match.group(1)
            year1 = match.group(2)
            year2 = match.group(3)
            date_suffix = f'{month}_{year1}_year_{year2}'
        else:
            date_suffix = date_part  # fallback
    else:
        # Handle single date format: 'Apr2023' -> '2023-04_monthly'
        # Dictionary to convert month abbreviations to numbers
        month_map = {
            'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
            'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08',
            'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
        }
        
        # Extract month and year
        match = re.match(r'([A-Za-z]+)(\d{4})', date_part)
        if match:
            month_abbr = match.group(1)
            year = match.group(2)
            
            # Get month number
            month_num = month_map.get(month_abbr, '00')
            date_suffix = f'{year}-{month_num}_monthly'
        else:
            date_suffix = date_part  # fallback
    
    # Build filename with direction if present
    if direction:
        cog_filename = f'{EVENT_NAME}_{direction}_{fsplit[0]}_{fsplit[1]}_{fsplit[3]}_{date_suffix}.tif'
    else:
        cog_filename = f'{EVENT_NAME}_{fsplit[0]}_{fsplit[1]}_{fsplit[3]}_{date_suffix}.tif'
    
    return cog_filename






filter_str = 'descending'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_desc(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2023-04_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2024-04_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_April_2017_year_2022.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2023-08_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2024-08_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_August_2017_year_2022.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2022-12_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2023-12_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_December_2017_year_2022.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2023-02_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2024-02_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Av

In [39]:
# # Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_desc, 
                                target_dir = "Sentinel-1/coherence", 
                                EVENT_NAME = EVENT_NAME)

Reading input: /tmp/tmp6bx_40ct_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmp1lzg4go_.tif


Testing filenames:
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2023-04_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2024-04_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_April_2017_year_2022.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2023-08_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2024-08_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_August_2017_year_2022.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2022-12_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2023-12_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_December_2017_year_2022.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2023-02_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2024-02_monthly.tif
  202410_Landslide_PalosVerdes_descending_PV_Avg1

Reading input: /tmp/tmpba9mdkdz_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmp16io44t1.tif


   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=0.9927435517311096, center sample non-zero=2035/3600
            Estimated data coverage: 55.6% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/coherence/202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2024-04_monthly.tif
   [MEMORY

Reading input: /tmp/tmptud6byhe_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmp39tp2gdo.tif


...

✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_October_2017_year_2022.tif

[35/37] Processing: drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_Sep2023_S1D71.tif
   Output filename: 202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2023-09_monthly.tif
   [MEMORY] Initial: 364.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=0.9904835820198059, center sample non-zero=2035/3600
            Estimated data coverage: 55.6% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF 

Reading input: /tmp/tmpeux5ny4t_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpz66gd62i.tif


   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=0.9969297051429749, center sample non-zero=2035/3600
            Estimated data coverage: 55.6% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/coherence/202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2024-09_monthly.tif
   [MEMORY] Final: 364.4 MB (Change: +0.0

Reading input: /tmp/tmp3me3pyec_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmp2jathek8.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_2024-09_monthly.tif

[37/37] Processing: drcs_activations/202410_Landslide_PalosVerdes/opera/coherence/descending/PV_Avg12dayCoh_September2017-2022_S1D71.tif
   Output filename: 202410_Landslide_PalosVerdes_descending_PV_Avg12dayCoh_S1D71_September_2017_year_2022.tif
   [MEMORY] Initial: 364.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=0.9924734234809875, center sample non-zero=2035/3600
            Estimated data coverage: 55.6% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing tempo

In [ ]:
# Display final results (for a single instance)
print(f"\n📊 Final Processing Results:")
print(f"Total files processed: {len(all_files_processed)}")
print(f"\nProcessed files DataFrame:")
all_files_processed

## Memory Usage Summary

You can check the final memory usage and cleanup

In [41]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 358.2 MB
  Available memory: 28677.6 MB
  Memory percent used: 9.3%


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.